## Imports

In [1]:
import sys
sys.executable # sanity check to make sure the env is correct

'/home/dave/anaconda3/bin/python'

In [2]:
import json
import pandas as pd
import numpy as np
import tqdm
from tqdm import tqdm
import ollama
from ollama import AsyncClient
from multiprocessing import Pool, cpu_count
import time
import re
import os
import asyncio

## Key Functions

In [18]:
# PROCESS DF -- SPLITS A DATAFRAME INTO n CHUNKS AND ASSIGNS m WORKERS TO PROCESS EACH CHUNK -- NEW
def process_df(df, num_chunks=200, workers=2, output_dir=''):
    print("Splitting df into chunks")
    chunks = np.array_split(df, num_chunks)
    work_items = []

    for chunk_id, chunk in enumerate(chunks):
        filename = f"chunk{chunk_id}.parquet"
        filepath = os.path.join(output_dir,filename)

        if os.path.exists(filepath):
            print(f"skipping chunk {chunk_id} (already exists)")
            continue

        work_items.append((chunk, chunk_id, output_dir))
    
    
    start_time = time.time()

    results = []
    with Pool(workers) as p:
        results = list(
            tqdm(
                p.imap(filter_chunk, work_items),
                total=len(work_items),
                desc="Filtering rows"
            )
        )
    end_time = time.time() - start_time
    print('processing tool:',end_time, ' seconds')
    
    return pd.concat(results, ignore_index=True)

In [141]:
# PROCESS DF -- SPLITS A DATAFRAME INTO n CHUNKS AND ASSIGNS m WORKERS TO PROCESS EACH CHUNK -- ORIGINAL
'''def process_df(df, num_chunks=200, workers=2):
    chunks = np.array_split(df, num_chunks)
    start_time = time.time()
    with Pool(workers) as p:
        results = list(
            tqdm(
                p.imap(filter_chunk, chunks),
                total=num_chunks,
                desc="Filtering rows"
            )
        )
    end_time = time.time() - start_time
    print('processing tool:',end_time, ' seconds')
    return pd.concat(results, ignore_index=True)'''

'def process_df(df, num_chunks=200, workers=2):\n    chunks = np.array_split(df, num_chunks)\n    start_time = time.time()\n    with Pool(workers) as p:\n        results = list(\n            tqdm(\n                p.imap(filter_chunk, chunks),\n                total=num_chunks,\n                desc="Filtering rows"\n            )\n        )\n    end_time = time.time() - start_time\n    print(\'processing tool:\',end_time, \' seconds\')\n    return pd.concat(results, ignore_index=True)'

In [16]:
# FILTER CHUNK -- EXECUTES A PROCESS ON EACH ITEM IN A SERIES -- NEW
async def filter_chunk_async(df_chunk, chunk_id,output_dir):
    chunk = df_chunk.copy()

    # PROCESS ROWS
    tasks = [process_row(row['description']) for _, row in chunk.iterrows()]
    #results = await asyncio.gather(*tasks)
    results = []
    for coro in tqdm(
        asyncio.as_completed(tasks),
        total=len(tasks),
        desc=f"Chunk {chunk_id}"
    ):
        result = await coro
        results.append(result)
    
    chunk['embeddings'] = results
    
    # NAME COMPLETED CHUNK AND SAVE AS CSV
    filename = f"chunk{chunk_id}.parquet"
    filepath = os.path.join(output_dir,filename)
    chunk.to_parquet(filepath, index=False)
    
    return chunk

def filter_chunk(args):
    df_chunk, chunk_id, output_dir = args
    return asyncio.run(filter_chunk_async(df_chunk, chunk_id, output_dir))

In [142]:
# FILTER CHUNK -- EXECUTES A PROCESS ON EACH ITEM IN A SERIES -- ORIGINAL
'''async def filter_chunk_async(df_chunk):
    chunk = df_chunk.copy()

    # PROCESS ROWS
    tasks = [process_row(row['description']) for _, row in chunk.iterrows()]
    results = await asyncio.gather(*tasks)
    chunk['embeddings'] = results
    
    return chunk

def filter_chunk(chunk):
    return asyncio.run(filter_chunk_async(chunk))'''

"async def filter_chunk_async(df_chunk):\n    chunk = df_chunk.copy()\n\n    # PROCESS ROWS\n    tasks = [process_row(row['description']) for _, row in chunk.iterrows()]\n    results = await asyncio.gather(*tasks)\n    chunk['embeddings'] = results\n\n    return chunk\n\ndef filter_chunk(chunk):\n    return asyncio.run(filter_chunk_async(chunk))"

In [12]:
# PROCESS ROW -- OPERATES ON ONE ITEM IN A SERIES
async def process_row(text):
    try:
        ollama = AsyncClient()
        response = await ollama.embeddings(model="mxbai-embed-large",prompt=text)        
        return response["embedding"]
        
    except Exception as e:
        return None


## Input DataFrame

In [13]:
df = pd.read_csv("/home/dave/dataCleanupUtility/target/clean_rows.csv")
df.head()

,Unnamed: 0,title,description
0,0,Glass Town,"A brief, fictionalized account of the daily li..."
1,1,Coretta Scott King,Explores the life and career of Coretta Scott ...
2,2,A tawdry place of salvation,In this first critical study of the work of Ja...
3,3,Inside COM,"""Microsoft's Component Object Model (COM) has ..."
4,4,Thunderbird gold,"While visiting his cousin in northern Canada, ..."


In [14]:
df = df[['title','description']]
df.head()

,title,description
0,Glass Town,"A brief, fictionalized account of the daily li..."
1,Coretta Scott King,Explores the life and career of Coretta Scott ...
2,A tawdry place of salvation,In this first critical study of the work of Ja...
3,Inside COM,"""Microsoft's Component Object Model (COM) has ..."
4,Thunderbird gold,"While visiting his cousin in northern Canada, ..."


## Testing

In [8]:
df_test = df.head(100)
df_test.head()

,title,description
0,Glass Town,"A brief, fictionalized account of the daily li..."
1,Coretta Scott King,Explores the life and career of Coretta Scott ...
2,A tawdry place of salvation,In this first critical study of the work of Ja...
3,Inside COM,"""Microsoft's Component Object Model (COM) has ..."
4,Thunderbird gold,"While visiting his cousin in northern Canada, ..."


In [8]:
process_df(df_test,workers=2, num_chunks=25)

NameError: name 'df_test' is not defined

## Benchmarking

In [129]:
def benchmark(df, max_workers=4, num_chunks=200):
    results = []
    outputs = []
    
    for n in range(1, max_workers +1):
        print(f"\nTesting with {n} worker(s)...")

        start = time.time()
        processed = process_df(df, workers=n, num_chunks=num_chunks)
        end = time.time()

        processed = processed.copy()
        processed['workers'] = n

        outputs.append(processed)
        
        results.append({
            "workers":n,
            "elapsed time": end - start
        })
    times_df = pd.DataFrame(results)
    outputs_df = pd.concat(outputs, ignore_index=True)
    return times_df, outputs_df

In [130]:
times_df, outputs = benchmark(df_test, max_workers =4, num_chunks=25)


Testing with 1 worker(s)...


/home/dave/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
Filtering rows: 100%|██████████| 25/25 [00:13<00:00,  1.86it/s]


processing tool: 14.167255878448486  seconds

Testing with 2 worker(s)...


/home/dave/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
Filtering rows: 100%|██████████| 25/25 [00:06<00:00,  3.79it/s]


processing tool: 7.675982713699341  seconds

Testing with 3 worker(s)...


/home/dave/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
Filtering rows: 100%|██████████| 25/25 [00:05<00:00,  4.70it/s]


processing tool: 6.680152416229248  seconds

Testing with 4 worker(s)...


/home/dave/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
Filtering rows: 100%|██████████| 25/25 [00:04<00:00,  5.45it/s]


processing tool: 6.103870630264282  seconds


In [92]:
times_df

,workers,elapsed time
0,1,9.849763
1,2,5.912042
2,3,5.141709
3,4,4.658275


In [97]:
outputs.groupby("workers").head()

,title,description,embeddings,workers
0,Glass Town,"A brief, fictionalized account of the daily li...","[0.1728658378124237, -0.33320003747940063, 0.2...",1
1,Coretta Scott King,Explores the life and career of Coretta Scott ...,"[-0.3468349575996399, -0.21785852313041687, -0...",1
2,A tawdry place of salvation,In this first critical study of the work of Ja...,"[-0.11076848208904266, -1.3954136371612549, -0...",1
3,Inside COM,"""Microsoft's Component Object Model (COM) has ...","[-0.4319303333759308, -0.4709661602973938, -0....",1
4,Thunderbird gold,"While visiting his cousin in northern Canada, ...","[0.2797992527484894, 0.21501488983631134, 0.27...",1
100,Glass Town,"A brief, fictionalized account of the daily li...","[0.1728658378124237, -0.33320003747940063, 0.2...",2
101,Coretta Scott King,Explores the life and career of Coretta Scott ...,"[-0.3468349575996399, -0.21785852313041687, -0...",2
102,A tawdry place of salvation,In this first critical study of the work of Ja...,"[-0.11076848208904266, -1.3954136371612549, -0...",2
103,Inside COM,"""Microsoft's Component Object Model (COM) has ...","[-0.4319303333759308, -0.4709661602973938, -0....",2
104,Thunderbird gold,"While visiting his cousin in northern Canada, ...","[0.2797992527484894, 0.21501488983631134, 0.27...",2


## Implementation

In [15]:
process_df(df,workers=2, num_chunks=1000)

Splitting df into chunks


/home/dave/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


skipping chunk 0 (already exists)
skipping chunk 1 (already exists)
skipping chunk 2 (already exists)
skipping chunk 3 (already exists)
skipping chunk 4 (already exists)
skipping chunk 5 (already exists)
skipping chunk 6 (already exists)
skipping chunk 7 (already exists)
skipping chunk 8 (already exists)
skipping chunk 9 (already exists)
skipping chunk 10 (already exists)
skipping chunk 11 (already exists)
skipping chunk 12 (already exists)
skipping chunk 13 (already exists)
skipping chunk 14 (already exists)
skipping chunk 15 (already exists)
skipping chunk 16 (already exists)
skipping chunk 17 (already exists)
skipping chunk 18 (already exists)
skipping chunk 19 (already exists)
skipping chunk 20 (already exists)
skipping chunk 21 (already exists)
skipping chunk 22 (already exists)
skipping chunk 23 (already exists)
skipping chunk 24 (already exists)
skipping chunk 25 (already exists)
skipping chunk 26 (already exists)
skipping chunk 27 (already exists)
skipping chunk 28 (already exi

Filtering rows: 100%|██████████| 64/64 [1:11:21<00:00, 66.90s/it]


processing tool: 4282.260821819305  seconds


,title,description,embeddings
0,Le pont,A complete French curriculum that provides a b...,None
1,The Mysteries of Udolpho,<p>The Mysteries of Udolpho (1794) is a Gothic...,None
2,America at war,Presents America's wartime history from the Re...,None
3,Philosophy of Mannerism,Sjoerd van Tuinen argues for the inseparabilit...,None
4,Prehistoric warfare in the American Southwest,"Most people today, including many archaeologis...",None
...,...,...,...
125115,NaN,Q/微信859034112办理毕业证成绩单丶使馆和教育部学历学位认证★诚招代理★\n英华教育...,"[0.20330633223056793, 0.4392056465148926, 0.82..."
125116,NaN,Books mentioned by Mercedes Shepard English in...,"[-0.040287502110004425, -0.44669461250305176, ..."
125117,NaN,"The noise of time, Theodosia. The Egyptian sta...","[-0.8145684599876404, 0.44507068395614624, -0...."
125118,NaN,Author databar,"[-0.5466983318328857, -0.6076515316963196, 0.1..."
